# T2.7 Indian Food — Baseline: ConvNeXt-V2-Base (full fine-tune, 384px)

ConvNeXt-V2-Base fully fine-tuned, pre-trained on ImageNet-22k/1k at 384-pixel resolution (timm: `convnextv2_base.fcmae_ft_in22k_in1k_384`). Multi-task head, MixUp/CutMix, mixed-precision — batch sized for Kaggle T4 (16 GB).

Runs locally under `/scratch3/jainil.bavishi/food_project` (paths match `dp_final_ss.ipynb`). Datasets are auto-downloaded via the Kaggle CLI on first run.

Datasets (downloaded automatically via Kaggle CLI; configure `~/.kaggle/kaggle.json` first):
1. `iamsouravbanerjee/indian-food-images-dataset` — 80 classes of dish images
2. `nehaprabhavalkar/indian-food-101` — metadata CSV (diet / course / region)

Mirrors the dataset / split / multi-task pipeline used in `dp_final.ipynb` so
results are directly comparable to the DINOv2-L LoRA headline model.


## 1. Imports

In [ ]:
# Imports + paths + reproducibility
import os, json, random, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

# All project artifacts live on /scratch3 (matches dp_final_ss.ipynb)
ROOT      = Path("/scratch3/jainil.bavishi/food_project")
DATA_RAW  = ROOT / "data/raw"
DATA_PROC = ROOT / "data/processed"
MODELS    = ROOT / "models"
RESULTS   = ROOT / "results"
for p in [DATA_RAW, DATA_PROC, MODELS, RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE_TYPE = "cuda" if DEVICE == "cuda" else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("ROOT  :", ROOT)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


## 2. Dependencies

In [ ]:
# Kaggle ships torch/torchvision/timm; install only if missing
import importlib, subprocess, sys
def ensure(pkg, pip_name=None):
    try: importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg])
ensure("timm")


## 3. Locate data

In [ ]:
# Locate / download data (auto-discover class dir + metadata CSV)
# Uses Kaggle CLI like dp_final_ss.ipynb. Set up ~/.kaggle/kaggle.json first.
img_dir = DATA_RAW / "indian-food-images"
if not img_dir.exists() or not any(img_dir.rglob("*.jpg")):
    !kaggle datasets download -d iamsouravbanerjee/indian-food-images-dataset -p {img_dir} --unzip
if not list(DATA_RAW.glob("indian_food*.csv")):
    !kaggle datasets download -d nehaprabhavalkar/indian-food-101 -p {DATA_RAW} --unzip

def find_class_root(root):
    best = None
    for p in root.rglob("*"):
        if p.is_dir():
            subs = [s for s in p.iterdir() if s.is_dir()]
            if len(subs) >= 50 and (best is None or len(subs) > best[1]):
                best = (p, len(subs))
    return best[0] if best else None

img_root = find_class_root(img_dir)
assert img_root is not None, f"No class dirs found under {img_dir}"
print("Image root:", img_root)

meta_csv = next(DATA_RAW.rglob("indian_food*.csv"), None)
assert meta_csv is not None, "indian_food*.csv not found in DATA_RAW"
print("Metadata  :", meta_csv)

classes = sorted([d.name for d in img_root.iterdir() if d.is_dir()])
print(f"Classes: {len(classes)}  e.g. {classes[:5]}")


## 4. Splits + aux labels

In [ ]:
# Splits + auxiliary (diet / course) labels
meta = pd.read_csv(meta_csv)
meta["name_norm"] = meta["name"].astype(str).str.lower().str.strip()

DIET_VOCAB   = ["vegan", "vegetarian", "non-vegetarian"]
COURSE_VOCAB = ["main course", "starter", "snack", "dessert"]

def parse_diet(s):
    if pd.isna(s): return -100
    s = str(s).lower()
    if "non" in s: return DIET_VOCAB.index("non-vegetarian")
    if "vegan" in s: return DIET_VOCAB.index("vegan")
    if "veg" in s: return DIET_VOCAB.index("vegetarian")
    return -100

def parse_course(s):
    if pd.isna(s): return -100
    s = str(s).lower()
    for i, k in enumerate(COURSE_VOCAB):
        if k.split()[0] in s: return i
    return -100

class_to_aux = {}
for c in classes:
    cn = c.lower().strip()
    row = meta[meta["name_norm"] == cn]
    if row.empty:
        row = meta[meta["name_norm"].str.contains(cn[:5], na=False, regex=False)]
    if not row.empty:
        r = row.iloc[0]
        class_to_aux[c] = {"diet": parse_diet(r.get("diet")),
                           "course": parse_course(r.get("course"))}
    else:
        class_to_aux[c] = {"diet": -100, "course": -100}
matched = sum(1 for v in class_to_aux.values() if v["diet"] != -100)
print(f"Aux match: {matched}/{len(classes)} classes have diet labels")

class_to_idx = {c: i for i, c in enumerate(classes)}
splits = {"train": [], "val": [], "test": []}
for c in classes:
    files = sorted(list((img_root/c).glob("*.[jJ][pP][gG]")) +
                   list((img_root/c).glob("*.[pP][nN][gG]")))
    random.Random(SEED).shuffle(files)
    n = len(files); n_tr = int(0.8*n); n_va = int(0.1*n)
    aux = class_to_aux[c]
    for i, p in enumerate(files):
        rec = {"path": str(p), "class": c, "label": class_to_idx[c],
               "diet": aux["diet"], "course": aux["course"]}
        if   i < n_tr:        splits["train"].append(rec)
        elif i < n_tr+n_va:   splits["val"].append(rec)
        else:                 splits["test"].append(rec)

(DATA_PROC/"splits.json").write_text(json.dumps({"classes": classes, **splits}))
print({k: len(v) for k, v in splits.items()})


## 5. Dataset + transforms + MixUp/CutMix

In [ ]:
# Dataset + transforms (matches dp_final.ipynb)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_transforms(split, img_size, profile="strong"):
    if split == "train":
        ops = [transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
               transforms.RandomHorizontalFlip()]
        if profile == "strong":
            ops.append(transforms.RandAugment(num_ops=2, magnitude=9))
        ops += [transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
        if profile == "strong":
            ops.append(transforms.RandomErasing(p=0.25))
        return transforms.Compose(ops)
    return transforms.Compose([
        transforms.Resize(int(img_size*1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

class IndianFoodDataset(Dataset):
    def __init__(self, split, img_size, profile="strong"):
        with open(DATA_PROC/"splits.json") as f:
            data = json.load(f)
        self.classes = data["classes"]
        self.records = data[split]
        self.transform = build_transforms(split, img_size, profile)
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        img = Image.open(r["path"]).convert("RGB")
        return self.transform(img), r["label"], r["diet"], r["course"]

def mixup_cutmix(x, y_dish, y_diet, y_course, alpha_mix=0.2, alpha_cut=1.0, prob=0.5):
    if random.random() > prob:
        return x, (y_dish, y_dish, 1.0), y_diet, y_course
    perm = torch.randperm(x.size(0), device=x.device)
    if random.random() < 0.5:
        lam = float(np.random.beta(alpha_mix, alpha_mix))
        x = lam*x + (1-lam)*x[perm]
    else:
        lam = float(np.random.beta(alpha_cut, alpha_cut))
        H, W = x.shape[2:]
        cut_h = int(H*math.sqrt(1-lam)); cut_w = int(W*math.sqrt(1-lam))
        cy, cx = np.random.randint(H), np.random.randint(W)
        y1, y2 = max(0, cy-cut_h//2), min(H, cy+cut_h//2)
        x1, x2 = max(0, cx-cut_w//2), min(W, cx+cut_w//2)
        x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
        lam = 1 - ((y2-y1)*(x2-x1) / float(H*W))
    return (x, (y_dish, y_dish[perm], lam),
            torch.full_like(y_diet, -100),
            torch.full_like(y_course, -100))


## 6. Model

In [ ]:
# Backbone + multi-task head — convnextv2_base.fcmae_ft_in22k_in1k_384
import timm

TIMM_NAME = "convnextv2_base.fcmae_ft_in22k_in1k_384"
IMG_SIZE  = 384
BATCH     = 12
EPOCHS    = 20
PATIENCE  = 4
LR        = 0.0002
WD        = 0.05

class MultiTaskHead(nn.Module):
    def __init__(self, dim, n_dish, n_diet=3, n_course=4, dropout=0.2):
        super().__init__()
        self.drop   = nn.Dropout(dropout)
        self.dish   = nn.Linear(dim, n_dish)
        self.diet   = nn.Linear(dim, n_diet)
        self.course = nn.Linear(dim, n_course)
    def forward(self, feat):
        feat = self.drop(feat)
        return self.dish(feat), self.diet(feat), self.course(feat)

class FoodCNN(nn.Module):
    def __init__(self, timm_name, n_dish):
        super().__init__()
        self.backbone = timm.create_model(timm_name, pretrained=True,
                                          num_classes=0, global_pool="avg")
        dim = self.backbone.num_features
        self.head = MultiTaskHead(dim, n_dish)
    def forward(self, x):
        feat = self.backbone(x)
        return self.head(feat)

def n_trainable(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
def n_total(m):     return sum(p.numel() for p in m.parameters())


## 7. Train

In [ ]:
# Train (mixed precision, MixUp/CutMix, multi-task loss, early stop)
train_ds = IndianFoodDataset("train", img_size=IMG_SIZE, profile="strong")
val_ds   = IndianFoodDataset("val",   img_size=IMG_SIZE, profile="basic")

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True,
                          persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

model = FoodCNN(TIMM_NAME, n_dish=len(train_ds.classes)).to(DEVICE)
print(f"Trainable: {n_trainable(model)/1e6:.1f}M / Total: {n_total(model)/1e6:.1f}M")

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, T_max=EPOCHS*len(train_loader), eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE_TYPE == "cuda"))

CKPT_PATH = MODELS / f"{TIMM_NAME.replace('.', '_')}_best.pt"

def train_one_epoch(model, loader, lam_diet=0.3, lam_course=0.2):
    model.train(); total = 0
    for x, y_dish, y_diet, y_course in tqdm(loader, leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y_dish = y_dish.to(DEVICE); y_diet = y_diet.to(DEVICE); y_course = y_course.to(DEVICE)
        x, (ya, yb, lam), y_diet, y_course = mixup_cutmix(x, y_dish, y_diet, y_course)
        opt.zero_grad()
        with torch.autocast(device_type=DEVICE_TYPE, dtype=torch.float16,
                            enabled=DEVICE_TYPE == "cuda"):
            l_dish, l_diet, l_course = model(x)
            loss_dish = lam*F.cross_entropy(l_dish, ya, label_smoothing=0.1) + \
                        (1-lam)*F.cross_entropy(l_dish, yb, label_smoothing=0.1)
            loss_diet   = (F.cross_entropy(l_diet,   y_diet,   ignore_index=-100)
                           if (y_diet   != -100).any() else torch.zeros((), device=DEVICE))
            loss_course = (F.cross_entropy(l_course, y_course, ignore_index=-100)
                           if (y_course != -100).any() else torch.zeros((), device=DEVICE))
            loss = loss_dish + lam_diet*loss_diet + lam_course*loss_course
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); scheduler.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = top5 = total = 0
    all_logits, all_labels = [], []
    for x, y, _, _ in tqdm(loader, leave=False):
        x = x.to(DEVICE); y = y.to(DEVICE)
        with torch.autocast(device_type=DEVICE_TYPE, dtype=torch.float16,
                            enabled=DEVICE_TYPE == "cuda"):
            l, _, _ = model(x)
        all_logits.append(l.float().cpu()); all_labels.append(y.cpu())
        correct += (l.argmax(1) == y).sum().item()
        top5    += (l.topk(5, 1).indices == y.unsqueeze(1)).any(1).sum().item()
        total   += y.size(0)
    return {"top1": correct/total, "top5": top5/total,
            "logits": torch.cat(all_logits), "labels": torch.cat(all_labels)}

best, since_best, history = 0.0, 0, {"train_loss": [], "val_top1": [], "val_top5": []}
for ep in range(1, EPOCHS+1):
    t0 = time.time()
    loss = train_one_epoch(model, train_loader)
    va   = evaluate(model, val_loader)
    history["train_loss"].append(loss)
    history["val_top1"].append(va["top1"]); history["val_top5"].append(va["top5"])
    improved = va["top1"] > best + 1e-4
    print(f"Ep {ep:02d}  loss={loss:.3f}  val_top1={va['top1']:.4f}  "
          f"val_top5={va['top5']:.4f}  ({time.time()-t0:.0f}s)"
          f"{'  *best*' if improved else f'  (no improve {since_best+1}/{PATIENCE})'}")
    if improved:
        best = va["top1"]; since_best = 0
        torch.save({"model": model.state_dict(), "classes": train_ds.classes,
                    "backbone": TIMM_NAME, "img_size": IMG_SIZE,
                    "epoch": ep, "val_top1": best}, CKPT_PATH)
    else:
        since_best += 1
        if since_best >= PATIENCE:
            print(f"Early stop @ epoch {ep}"); break

history["best_val_top1"] = best
(RESULTS / f"{TIMM_NAME.replace('.', '_')}_history.json").write_text(json.dumps(history))
print(f"Best val top-1: {best:.4f}")


## 8. Evaluate test set

In [ ]:
# Test-set evaluation + confusion matrix
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score

ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model"])

test_ds = IndianFoodDataset("test", img_size=IMG_SIZE, profile="basic")
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2)

te = evaluate(model, test_loader)
probs  = F.softmax(te["logits"], dim=1).numpy()
labels = te["labels"].numpy()
preds  = probs.argmax(1)

top1 = (preds == labels).mean()
top5 = top_k_accuracy_score(labels, probs, k=5,
                            labels=list(range(len(test_ds.classes))))
print(f"TEST  top-1: {top1:.4f}   top-5: {top5:.4f}")
print(classification_report(labels, preds,
                            target_names=test_ds.classes, zero_division=0)[:3000])

cm = confusion_matrix(labels, preds)
np.save(RESULTS / f"{TIMM_NAME.replace('.', '_')}_confusion_matrix.npy", cm)

(RESULTS / f"{TIMM_NAME.replace('.', '_')}_test_metrics.json").write_text(json.dumps({
    "backbone": TIMM_NAME, "img_size": IMG_SIZE,
    "test_top1": float(top1), "test_top5": float(top5),
    "best_val_top1": float(ckpt.get("val_top1", 0.0)),
    "trainable_params_M": n_trainable(model) / 1e6,
}, indent=2))
print("Saved metrics to", RESULTS)


## 9. Confusion matrix

In [ ]:
# Confusion-matrix figure + top-10 confusions
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, xticklabels=test_ds.classes, yticklabels=test_ds.classes,
            cmap="Blues", cbar=True, ax=ax, square=True)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"{TIMM_NAME} — confusion matrix (test)")
plt.xticks(rotation=90, fontsize=6); plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig(RESULTS / f"{TIMM_NAME.replace('.', '_')}_confusion_matrix.png", dpi=120)
plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
flat = [(test_ds.classes[i], test_ds.classes[j], int(off[i, j]))
        for i in range(len(test_ds.classes)) for j in range(len(test_ds.classes))]
flat.sort(key=lambda t: -t[2])
print("Top confusions (true -> pred, count):")
for t in flat[:10]:
    print(f"  {t[0]:30s} -> {t[1]:30s}  {t[2]}")
